<a href="https://colab.research.google.com/github/mooch443/dataset-fixer/blob/main/notebooks/01_controlled_splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Controlled, group-aware dataset splitting

This tutorial demonstrates `Dataset.split()`: freezing physically related frames into groups, assigning those groups deterministically, previewing the proposal, and inspecting the resulting provenance.

> **AI-generation disclosure:** this project and tutorial are largely AI-generated under human direction and review. Independently validate results for your data.

The example uses a pinned subset of the public [SAWIT camera-trap dataset](https://github.com/dtnguyen0304/sawit), including its official images and YOLO labels. SAWIT declares the [MIT License](https://github.com/dtnguyen0304/sawit/blob/main/LICENSE).

## 1. Install the package

The cell is idempotent in a fresh Colab runtime. Restart the runtime only if Colab asks after changing preinstalled dependencies.

In [ ]:
import importlib, os, subprocess, sys, tempfile
from pathlib import Path

repo = next((candidate for candidate in [Path('/content/dataset-fixer'), Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / 'pyproject.toml').is_file()), None)
if repo is None:
    repo = Path(tempfile.mkdtemp(prefix='dataset-fixer-source-')) / 'dataset-fixer'
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/mooch443/dataset-fixer.git', str(repo)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)])
sys.path.insert(0, str(repo / 'src'))
sys.path.insert(0, str(repo))
importlib.invalidate_caches()
import dataset_fixer
print('dataset-fixer:', dataset_fixer.__version__, dataset_fixer.__file__)
WORK_ROOT = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir()) / 'dataset-fixer-notebooks'
WORK_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Download and inspect the public MIT-licensed example data

The downloader fetches official SAWIT files directly from a pinned upstream commit. `SOURCE.json` records the source URL, commit, selection rule, license, and SHA-256 values. The tutorial does not generate or alter pixels or labels.

In [ ]:
from dataset_fixer import Dataset
from examples.download_public_examples import download_sawit_examples

example_root = WORK_ROOT / 'dataset-fixer-public-examples'
paths = download_sawit_examples(example_root, images_per_class=8)
dataset = Dataset.open(paths['unsplit'], task='detect', deep=True)
print(dataset)
print('classes:', dataset.classes)
print((example_root / 'SOURCE.json').read_text()[:800])
dataset.visualize(split='train', n=9, seed=42, columns=3)

## 3. Split by physical sequence

`group_by` receives each source image path. The callback result becomes an indivisible allocation unit. For demonstration, adjacent numbered files within each SAWIT category are paired. Replace this rule with a real capture session, video, site, subject, or other physical identifier for your data. A local seeded RNG makes the result reproducible, and `visualize=True` creates both the pre-operation sanity check and final distribution audit.

In [ ]:
import shutil
import re

def source_group(path):
    number = int(re.search(r'(\d+)', path.stem).group(1))
    return path.parent.name, number // 2

split_destination = WORK_ROOT / 'sawit-grouped-split'
if split_destination.exists():
    shutil.rmtree(split_destination)

planned_split = dataset.split(
    {'train': 0.70, 'val': 0.20, 'test': 0.10},
    group_by=source_group,
    seed=42,
    visualize=True,
)
print(planned_split)
assert planned_split.data_yaml is None and not split_destination.exists()
split_dataset = planned_split.export(destination=split_destination)
print('name:', split_dataset.name)
print('location:', split_dataset.location)
print('data.yaml:', split_dataset.data_yaml)
print('splits:', split_dataset.splits)
print('training ready:', split_dataset.training_ready)

## 4. Verify the grouping and reproduction record

`split()` only creates an immutable in-memory plan. `export()` is the sole write boundary. The assertion below verifies that each physical group maps to exactly one output split; the manifest records the seed, resolved callback outputs, package revision, environment, and source fingerprint.

In [ ]:
import json
from collections import defaultdict

group_splits = defaultdict(set)
for record in split_dataset.provenance.values():
    group = source_group(Path(record['parent_image']))
    group_splits[group].add(record['output_split'])
assert all(len(value) == 1 for value in group_splits.values())
print({group: next(iter(value)) for group, value in sorted(group_splits.items())})

manifest = json.loads((split_dataset.location / 'dataset-fixer.json').read_text())
print('settings fingerprint:', manifest['settings_fingerprint'])
print('tool revision:', manifest['environment']['dataset_fixer_git'])
print('provenance rows:', len(split_dataset.provenance))

### What to adapt for real data

- Change `group_by` to extract your orchard row, video, site, animal, patient, or sampling session from the path.
- Use `assign=` when specific groups have predetermined destinations. Conflicting assignments within one group fail before output is written.
- Use `deep=True` in `Dataset.open()` when byte-identical cross-split duplicates must also be detected.